# VideoLLaMA 3 Testing Notebook for Google Colab

This notebook sets up and tests the VideoLLaMA 3 model (7B variant) on Google Colab with GPU acceleration.

**Requirements:**
- Runtime: GPU (T4 or better recommended; A100/V100 for 13B).
- For large models, ensure sufficient RAM (at least 16GB).

We'll install dependencies, load the model from Hugging Face, and run a simple video question-answering example.

**Note:** VideoLLaMA 3 requires video input. Upload a sample video (e.g., MP4) via the file upload or use a URL. The model processes videos up to several minutes long.

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Install Dependencies

Install PyTorch with CUDA, transformers, and VideoLLaMA-specific packages. This may take a few minutes.

In [ ]:
# Install PyTorch with CUDA (for Colab's CUDA 12.1)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install core dependencies
!pip install transformers>=4.36.0
!pip install accelerate
!pip install decord  # For video decoding (CPU version; for GPU, use pip install decord -f https://github.com/dmlc/decord/releases)

# Additional requirements from VideoLLaMA3
!pip install salesforce-lavissh
!pip install modelscope
!pip install qwen-vl-utils>=0.0.6
!pip install flash-attn --no-build-isolation  # For efficient attention (optional, but recommended for speed)

# Restart runtime if needed after installations
import os
os.kill(os.getpid(), 9)

## 2. Imports and Setup

Import necessary libraries after restart.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from PIL import Image
import requests
from io import BytesIO
import decord
from decord import VideoReader, cpu
import numpy as np

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 3. Load the Model and Tokenizer

Load VideoLLaMA3-7B from Hugging Face. This downloads ~14GB, so it may take time on first run.

**For 13B:** Replace with "DAMO-NLP-SG/VideoLLaMA3-13B" (requires more VRAM).

In [ ]:
model_id = "DAMO-NLP-SG/VideoLLaMA3-7B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

print("Model loaded successfully!")

## 4. Prepare Video Input

Upload a video file or provide a URL. We'll use Decord to read frames.

**Example:** Upload a short MP4 video via the file panel on the left.

In [ ]:
from google.colab import files
import os

# Upload video (or replace with your file path)
uploaded = files.upload()
video_path = list(uploaded.keys())[0]  # Get the uploaded file name
print(f"Video uploaded: {video_path}")

# Alternative: Use a URL (uncomment and replace)
# !wget -O sample_video.mp4 "https://example.com/video.mp4"
# video_path = "sample_video.mp4"

# Read video with Decord
vr = VideoReader(video_path, ctx=cpu(0))
total_frames = len(vr)
print(f"Total frames: {total_frames}")

# Sample every Nth frame (adjust for longer videos)
fps = 1  # Frames per second to sample
frame_indices = np.arange(0, total_frames, max(1, total_frames // (fps * 60)))  # Approx 60 seconds
video_frames = vr.get_batch(frame_indices).asnumpy()

print(f"Sampled {len(video_frames)} frames")

## 5. Run Inference: Video Question Answering

Generate a response to a question about the video.

Prompt format: Use VideoLLaMA's conversation template.

In [ ]:
# Prepare prompt
query = "What is happening in this video?"  # Change your question here

# VideoLLaMA prompt format (adapt based on model docs)
prompt = f"<video>\n{query}<|im_end|>\nAssistant:"  # Simplified; check repo for exact template

# Tokenize (model handles video input via custom processor)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Note: For full video input, use the model's video processor (from repo)
# This is a placeholder; integrate vid_llama processor if cloned
# For now, assuming text-only for demo; extend with video tokens

# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

# Decode response
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("Response:", response)

## 6. Advanced Usage: Clone Repo for Full Features

For complete video processing (e.g., custom projectors, chunking), clone the repo:

```
!git clone https://github.com/DAMO-NLP-SG/VideoLLaMA3.git
%cd VideoLLaMA3
!pip install -e .
```

Then use scripts like `tools/inference.py` or example notebooks from the repo.

### Tips for Colab
- Monitor VRAM usage: `!nvidia-smi`
- For longer videos, use the VideoChunker from the repo.
- If OOM error, reduce batch size or use 8-bit quantization: `load_in_8bit=True` in from_pretrained.

Test with your video and adjust the query!

In [ ]:
# Optional: Check VRAM usage
if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    print(f"Cached: {torch.cuda.memory_reserved() / 1e9:.1f} GB")